In [ ]:
# imports
import json

import duckdb
import pandas as pd

pd.set_option("display.max_rows", None)

In [ ]:
# run sql
def run_sql(sql: str):
    with duckdb.connect(str(DB_PATH), read_only=True) as con:
        return con.sql(sql).df()

In [ ]:
# run row
def run_row(sql: str):
    with duckdb.connect(str(DB_PATH), read_only=True) as con:
        return con.sql(sql).fetchone()

In [ ]:
# list schemas
def show_schemas():
    return run_sql("""
        select schema_name
        from information_schema.schemata
        where catalog_name = current_database()
          and schema_name not in ('information_schema', 'pg_catalog')
        order by 1
    """)

In [ ]:
# list tables
def show_tables(schema: str):
    df = run_sql(f"""
        select t.table_name as name, t.table_type as type, count(*) as columns
        from information_schema.tables as t
        join information_schema.columns as c
          on t.table_schema = c.table_schema and t.table_name = c.table_name
        where t.table_schema = '{schema}'
        group by 1, 2
        order by 1
    """)
    df["rows"] = [run_row(f'select count(*) from {schema}."{name}"')[0] for name in df["name"]]
    return df

In [ ]:
# list columns
def show_columns(schema: str):
    return run_sql(f"""
        select table_name, ordinal_position as pos, column_name, data_type
        from information_schema.columns
        where table_schema = '{schema}'
        order by table_name, ordinal_position
    """)

In [ ]:
# profile columns
def profile_columns(schema: str, table: str = None):
    parts = []
    columns = show_columns(schema)
    if table:
        columns = columns[columns["table_name"] == table]
    for r in columns.itertuples():
        col = f'"{r.column_name}"'
        parts.append(f"""
            select {r.pos} as pos, '{r.table_name}' as table_name, '{r.column_name}' as column_name,
                   '{r.data_type}' as type, count(*) as rows, count(*) - count({col}) as nulls,
                   count(distinct {col}) as distinct_values,
                   min({col})::varchar as min, max({col})::varchar as max
            from {schema}.{r.table_name}
        """)
    df = run_sql(" union all ".join(parts)).sort_values(["table_name", "pos"], ignore_index=True)
    df["null_pct"] = (df["nulls"] / df["rows"] * 100).round(1)
    return df.drop(columns="pos")

In [ ]:
# id candidates
def find_keys(prof):
    is_id = prof["column_name"].str.endswith("_id")
    unique = (prof["nulls"] == 0) & (prof["distinct_values"] == prof["rows"])
    return prof[is_id & unique][["table_name", "column_name", "type", "rows"]].reset_index(drop=True)

In [ ]:
# dbt tests
def show_tests(schema: str):
    nodes = json.loads((ROOT / "src/dbt/target/manifest.json").read_text(encoding="utf-8"))["nodes"]
    rows = []
    for node in nodes.values():
        meta = node.get("test_metadata")
        if node["resource_type"] != "test" or not meta or nodes[node["attached_node"]]["schema"] != schema:
            continue
        kwargs = meta["kwargs"]
        target = f"{kwargs['to']}.{kwargs['field']}" if meta["name"] == "relationships" else ""
        rows.append({
            "model": nodes[node["attached_node"]]["name"],
            "column": node.get("column_name"),
            "test": meta["name"],
            "to": target,
        })
    return pd.DataFrame(rows).sort_values(["model", "column", "test"], ignore_index=True)

In [ ]:
# declared keys
def show_constraints():
    return run_sql("""
        select schema_name, table_name, constraint_type, constraint_column_names
        from duckdb_constraints()
        order by 1, 2
    """)

In [ ]:
# sample rows
def show_sample(schema: str, table: str, n: int = 5):
    return run_sql(f"select * from {schema}.{table} limit {n}")

In [ ]:
# top values
def count_values(schema: str, table: str, column: str, n: int = 10):
    return run_sql(f"""
        select "{column}" as value, count(*) as rows,
               round(100.0 * count(*) / sum(count(*)) over (), 1) as pct
        from {schema}.{table}
        group by 1
        order by 2 desc
        limit {n}
    """)

In [ ]:
# show nulls
def show_nulls(schema: str):
    parts = []
    for r in show_columns(schema).itertuples():
        col = f'"{r.column_name}"'
        parts.append(f"""
            select '{r.table_name}' as table_name, '{r.column_name}' as column_name,
                   count(*) filter (where {col} is null) as nulls, count(*) as rows
            from {schema}.{r.table_name}
        """)
    df = run_sql(" union all ".join(parts))
    df["null_pct"] = (df["nulls"] / df["rows"] * 100).round(1)
    return df[df["nulls"] > 0].reset_index(drop=True)

In [ ]:
# check nulls
def check_nulls(schema: str, table: str, columns: list):
    rows = []
    for column in columns:
        nulls = run_row(f'select count(*) filter (where "{column}" is null) from {schema}.{table}')[0]
        rows.append((table, column, nulls, "OK" if nulls == 0 else "FAIL"))
    return pd.DataFrame(rows, columns=["table", "column", "nulls", "status"])

In [ ]:
# check text
def check_text(schema: str):
    rows = []
    for table, group in show_columns(schema).query("data_type == 'VARCHAR'").groupby("table_name"):
        edge = extra = empty = 0
        for column in group["column_name"]:
            col = f'"{column}"'
            e, x, m = run_row(f"""
                select count(*) filter (where {col} <> trim({col})),
                       count(*) filter (where regexp_matches({col}, '\\s\\s|[^\\S ]')),
                       count(*) filter (where {col} = '')
                from {schema}.{table}
            """)
            edge, extra, empty = edge + e, extra + x, empty + m
        status = "OK" if edge + extra + empty == 0 else "FAIL"
        rows.append((table, len(group), edge, extra, empty, status))
    return pd.DataFrame(rows, columns=["table", "text_columns", "edge_spaces", "extra_spaces", "empty_strings", "status"])

In [ ]:
# one table
def show_table(schema: str, table: str):
    display(show_sample(schema, table))
    display(profile_columns(schema, table))
    tests = show_tests(schema)
    display(tests[tests["model"] == table])